# 09 — PSMA-PET-CT-Lesions: organ segmentation with TotalSegmentator (Colab GPU)

Organ masks for the PBPK compartments and for the dosimetry of the virtual cohort, on every study of PSMA-PET-CT-Lesions. The study list and tracer come from the inventory of notebook 08.

**Runtime.** Google Colab with a GPU, run from VS Code through the Colab extension. Google Drive is mounted with `drive.mount` (in VS Code, the command *Colab: Mount Google Drive to Server* inserts the same snippet). Run the cells in order:
1. setup;
2. test on 3 studies;
3. full run, which can be re-run after a disconnection until nothing is left;
4. QC.

**Input.**
- CT channel `*_0000.nii.gz` of `imagesTr`: the CT already resampled to the PET grid. Its voxels are on the same grid as the PET (`_0001`) and the lesion label, which is the grid the network uses.
- Location on Drive: `<DRIVE_ROOT>/psma_ct/`, searched recursively.
- `psma_inventory.csv` (from `data/processed/psma/`), copied to `<DRIVE_ROOT>/`.

**Tasks and classes** (TotalSegmentator 2.18.0; names checked against the `class_map` of the installed package in the setup cell).

| output | TotalSegmentator task | classes kept | PBPK use |
|---|---|---|---|
| `organs` | `total` with `roi_subset` | `kidney_left`, `kidney_right`, `liver`, `spleen`, `urinary_bladder`, `small_bowel`, `duodenum`, `colon`, `stomach`, `heart`, `aorta`, `vertebrae_C1` … `vertebrae_L5`, `vertebrae_S1`, `sacrum` | kidneys, liver, spleen, bladder, GI tract, blood pool (heart, aorta), red-marrow surrogate (vertebrae) |
| `salivary` | `head_glands_cavities` (full task, 19 classes) | `parotid_gland_left`, `parotid_gland_right`, `submandibular_gland_left`, `submandibular_gland_right` | salivary glands |

Notes on the class choice:
- `duodenum` is a separate class in TotalSegmentator; it is added so that the small-bowel compartment can include it.
- `heart` in the `total` task is the whole heart (myocardium and chambers): a blood-pool VOI must be eroded or taken from the aorta lumen. The heart chambers are in `heartchambers_highres`, which requires a licence.
- The vertebrae are kept as separate labels, so that a red-marrow surrogate can use L2-L4, T/L or the whole spine. C1-C2 also serve as a field-of-view marker for the salivary glands in the QC.
- Other classes (e.g. `prostate`, `inferior_vena_cava`, `pancreas`) can be appended to `ORGAN_CLASSES`. Already processed studies would then have to be redone: delete the `organs` folder.

**Licence.**
- `total` and `head_glands_cavities` are listed in the README under "Openly available for any usage (Apache-2.0 license)". Neither is among the licensed (`commercial_models`) tasks; the setup cell checks this with `requires_license`. The parotid and submandibular glands are therefore freely available.
- To cite for the glands: Cancers 2024;16(2):415 (link in the README).
- **Limitation of the parotid model.** The gland group of that paper is submandibular left and right and thyroid: the parotid model has no per-structure published validation there, and the 0.75 × 0.75 × 1 mm resolution comes from TotalSegmentator, not from the paper. Only our QC supports it. Complete parotids in the phantoms of notebook 11 have a median of 12.4 mL per gland against 19.1 g (Okamoto) and 24.98 mL (Scarpa), that is 35-50% lower, so parotid volumes and totals are underestimated.
- Not available in any task: the **lacrimal glands** (declared as a limitation). The sublingual glands are also absent.
- `--roi_subset` works only with `total`/`total_mr` (ValueError otherwise), so the head task runs in full, and only the four glands are kept.
- `head_glands_cavities` crops around the `skull` found by a first coarse model. If no skull is in the field of view the study fails, and the failure is logged.

**Resolution caveat.**
- The CT is at the PET resolution: about 4.07 × 4.07 × 2-5 mm, or 2.73 × 2.73 × 3.27 mm, depending on the scanner (notebook 08).
- TotalSegmentator `total` is trained at 1.5 mm, and `head_glands_cavities` at 0.75 × 0.75 × 1 mm. The input is upsampled to the model resolution and the result resampled back to the PET grid.
- Large organs (liver, kidneys, spleen) are not limited by this. Small or thin structures need visual QC: salivary glands (a submandibular gland of ~8 mL is ~120 voxels at 4 × 4 × 4 mm), duodenum, small bowel, and vertebrae near the field-of-view edges.
- The CT is mostly contrast-enhanced (571/597), which TotalSegmentator handles. For Monte Carlo the densities are overridden anyway.
- `ROBUST_CROP = True` uses the 3 mm instead of the 6 mm model for the `roi_subset` crop. The README warns that the 6 mm crop can cut structures off, and at PET resolution the input is already coarse.
- `FAST_ORGANS` selects the model of the organ task: `False` (default) is the standard 1.5 mm model, an ensemble of 5 nnU-Nets; `True` is the `--fast` 3 mm model (a single network, task 297).
  - The 3 mm model is closer to the native resolution of these CTs and faster. Section 4b compares the two on the test studies (Dice and volume per organ) before the full run.
  - The salivary task has no fast model (`disallow_fast`) and is unchanged.
  - The choice is stored in `labelmap_organs.json`. The setup cell stops if it is changed after organ masks already exist, so the cohort is never a mix of the two models.

**Outputs on Drive** (`<DRIVE_ROOT>/psma_segmentations/`):
- `organs/<study_id>.nii.gz` and `salivary/<study_id>.nii.gz`: uint8 multilabel on the CT/PET grid (same shape and affine as `_0000`);
- `labelmap_organs.json`, `labelmap_salivary.json`: label value → class name, TotalSegmentator source index, version and options (including `fast`);
- `compare_fast/`: the test studies segmented with the other organ model, `fast_vs_standard.csv` and `timing.csv` (section 4b);
- `run_log.csv`: one row per attempt (study, task, status, seconds, labels present, GPU, error);
- `errors/<study_id>__<task>.txt`: traceback of each failure;
- `qc/qc_volumes.csv`, `qc/qc_flags.csv`, `qc/overlays_*.png`;
- salivary QC: `qc/qc_salivary_glands.csv` (one row per gland and study: volume, present, truncated, gap to the top slice), `qc/salivary_vs_literature.csv`, `qc/salivary_volumes_vs_literature.png`, `qc/salivary_<study_id>.png` for the test studies.

**Resume rules.**
- A study and task is done when its final file exists. Every file is written to a temporary name and renamed at the end, so a disconnection never leaves a partial file under the final name.
- Studies with an error file are skipped unless `RETRY_ERRORS = True`.
- A failure is logged and the loop goes on.

**Order.** The 68Ga studies first (the PBPK cohort), then 18F; within each tracer, the order of `study_id`.

**About this notebook.** TotalSegmentator organs and salivary glands on the PSMA CTs, QC.
- **Inputs:** CTs on Google Drive (`<DRIVE_ROOT>/psma_ct/`).
- **Outputs:** `$LU177_DATASET/PSMA-PET-CT-Lesions_v3/psma_segmentations/` (copied from Drive).
- **Run time:** several Colab GPU sessions.
- **Environment:** Colab (GPU), not runnable locally.

## 1. Setup

Mount Drive, check the GPU, install TotalSegmentator. Colab already has PyTorch with CUDA. The model weights are downloaded at the first prediction of each session, into `TOTALSEG_HOME_DIR` on the local disk, which is faster than reading them from Drive.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PATH = Path(os.environ.get("LU177_DRIVE_PATH", "/content/drive/MyDrive/lu177"))
    LOCAL_TMP = Path("/content/tmp_totalseg")
    os.environ["TOTALSEG_HOME_DIR"] = "/content/totalseg_home"
else:
    DRIVE_PATH = Path(os.environ["LU177_DRIVE_PATH"])
    LOCAL_TMP = Path(os.environ.get("LU177_LOCAL_TMP", "/tmp/tmp_totalseg"))

CT_DIR = DRIVE_PATH / "psma_ct"
OUT_DIR = DRIVE_PATH / "psma_segmentations"
INVENTORY_CSV = DRIVE_PATH / "psma_inventory.csv"
print(DRIVE_PATH, "exists:", DRIVE_PATH.exists(), "| CT folder exists:", CT_DIR.exists(),
      "| inventory exists:", INVENTORY_CSV.exists())

In [ ]:
%pip install -q "TotalSegmentator==2.18.0"
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
import csv
import json
import shutil
import tempfile
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from totalsegmentator.config import set_config_key, setup_totalseg
from totalsegmentator.map_to_binary import class_map
from totalsegmentator.python_api import totalsegmentator
from totalsegmentator.registry import package_version, requires_license

setup_totalseg()
set_config_key("send_usage_stats", False)
TS_VERSION = package_version()
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
print(f"TotalSegmentator {TS_VERSION}, torch {torch.__version__}, GPU: {GPU}")
if DEVICE == "cpu":
    print("WARNING: no GPU, the run will be very slow (Runtime > Change runtime type > GPU).")

**Classes and label maps.** The output values are contiguous (1…N) in the order of the lists below. The TotalSegmentator index of each class is stored in the JSON label map. The cell stops if a name is not a class of the installed task, or if a task requires a licence.

In [ ]:
VERTEBRAE = ([f"vertebrae_C{i}" for i in range(1, 8)] + [f"vertebrae_T{i}" for i in range(1, 13)]
             + [f"vertebrae_L{i}" for i in range(1, 6)] + ["vertebrae_S1", "sacrum"])
ORGAN_CLASSES = ["kidney_left", "kidney_right", "liver", "spleen", "urinary_bladder", "small_bowel", "duodenum",
                 "colon", "stomach", "heart", "aorta"] + VERTEBRAE
SALIVARY_CLASSES = ["parotid_gland_left", "parotid_gland_right", "submandibular_gland_left", "submandibular_gland_right"]
ROBUST_CROP = True
FAST_ORGANS = False
RETRY_ERRORS = True
TRUNC_MARGIN_SLICES = 2

TASKS = {
    "organs": {"ts_task": "total", "classes": ORGAN_CLASSES, "roi_subset": ORGAN_CLASSES, "fast": FAST_ORGANS},
    "salivary": {"ts_task": "head_glands_cavities", "classes": SALIVARY_CLASSES, "roi_subset": None, "fast": False},
}
for name, t in TASKS.items():
    inverse = {v: k for k, v in class_map[t["ts_task"]].items()}
    missing = [c for c in t["classes"] if c not in inverse]
    assert not missing, f"{name}: not classes of task {t['ts_task']}: {missing}"
    assert not requires_license(t["ts_task"]), f"{t['ts_task']} requires a licence"
    t["source_index"] = {c: int(inverse[c]) for c in t["classes"]}
    t["lut"] = np.zeros(256, dtype=np.uint8)
    for k, c in enumerate(t["classes"], start=1):
        t["lut"][t["source_index"][c]] = k
    print(f"{name}: task {t['ts_task']}, fast={t['fast']}, {len(t['classes'])} classes kept, licence required: {requires_license(t['ts_task'])}")

for sub in ["organs", "salivary", "errors", "qc"]:
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
LOCAL_TMP.mkdir(parents=True, exist_ok=True)
for name, t in TASKS.items():
    lm_path = OUT_DIR / f"labelmap_{name}.json"
    if lm_path.exists() and any((OUT_DIR / name).glob("*.nii.gz")):
        previous = json.loads(lm_path.read_text())
        if previous.get("fast", False) != t["fast"] or previous.get("roi_subset") != t["roi_subset"]:
            raise RuntimeError(f"{name}: existing masks were made with fast={previous.get('fast', False)} and a "
                               f"different setting is requested now; move or delete {OUT_DIR / name} first")
    lm_path.write_text(json.dumps({
        "task": t["ts_task"], "fast": t["fast"], "model_resolution_mm": (3.0 if t["fast"] else 1.5) if name == "organs" else [0.75, 0.75, 1.0],
        "roi_subset": t["roi_subset"], "robust_crop": ROBUST_CROP if t["roi_subset"] else None,
        "totalsegmentator_version": TS_VERSION, "background": 0,
        "labels": {str(k): c for k, c in enumerate(t["classes"], start=1)},
        "totalsegmentator_index": t["source_index"],
        "grid": "same shape and affine as the input CT imagesTr/<study_id>_0000.nii.gz (PET grid)"}, indent=2))

## 2. Study list and order

In [ ]:
ct_files = {p.name.removesuffix("_0000.nii.gz"): p for p in sorted(CT_DIR.rglob("*_0000.nii.gz"))}
inventory = pd.read_csv(INVENTORY_CSV)
inventory = inventory[inventory.study_id.isin(ct_files)].copy()
inventory["tracer_order"] = inventory.tracer.map({"68Ga": 0, "18F": 1})
inventory = inventory.sort_values(["tracer_order", "study_id"])
ORDER = list(inventory.study_id)
print(f"CT files on Drive: {len(ct_files)}; in the inventory: {len(ORDER)} "
      f"(68Ga {int((inventory.tracer == '68Ga').sum())}, 18F {int((inventory.tracer == '18F').sum())})")
not_in_inventory = sorted(set(ct_files) - set(ORDER))
if not_in_inventory:
    print("CT files without inventory row (not processed):", not_in_inventory[:10], "...")
inventory.groupby("tracer")[["dx_mm", "dz_mm"]].agg(lambda s: sorted(s.round(2).unique()))

## 3. Segmentation function and resumable loop

For each study and task:
1. copy the CT from Drive to the local disk;
2. run TotalSegmentator (multilabel);
3. check that the result is on the CT grid and contains only the expected source indices;
4. remap to the contiguous labels;
5. save locally, copy to Drive under a temporary name `.part`, and rename to the final name.

Each attempt appends a row to `run_log.csv`. A failure writes the traceback to `errors/` and the loop continues. `torch.cuda.empty_cache()` after each study limits GPU memory build-up over long runs.

In [ ]:
LOG_CSV = OUT_DIR / "run_log.csv"
LOG_FIELDS = ["timestamp_utc", "study_id", "tracer", "task", "status", "seconds", "labels_present", "n_labels",
              "gpu", "ts_version", "error"]


def final_path(study_id, task):
    return OUT_DIR / task / f"{study_id}.nii.gz"


def error_path(study_id, task):
    return OUT_DIR / "errors" / f"{study_id}__{task}.txt"


def log(row):
    new = not LOG_CSV.exists()
    with open(LOG_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in LOG_FIELDS})


def atomic_copy(src, dst):
    part = dst.with_name(dst.name + ".part")
    shutil.copyfile(src, part)
    os.replace(part, dst)


def segment(study_id, task, fast=None, dest=None):
    t = TASKS[task]
    fast = t["fast"] if fast is None else fast
    dest = final_path(study_id, task) if dest is None else dest
    with tempfile.TemporaryDirectory(dir=LOCAL_TMP) as tmp:
        tmp = Path(tmp)
        ct_local = tmp / "ct.nii.gz"
        shutil.copyfile(ct_files[study_id], ct_local)
        ct = nib.load(ct_local)
        seg = totalsegmentator(ct_local, tmp / "seg_ts.nii.gz", ml=True, task=t["ts_task"], roi_subset=t["roi_subset"],
                               fast=fast, robust_crop=bool(ROBUST_CROP and t["roi_subset"]), device=DEVICE,
                               quiet=True, nr_thr_saving=1)
        data = np.asarray(seg.dataobj).astype(np.uint8)
        if data.shape != ct.shape or not np.allclose(seg.affine, ct.affine, atol=1e-3):
            raise RuntimeError(f"output grid {data.shape} differs from CT grid {ct.shape}")
        values = set(np.unique(data).tolist()) - {0}
        if task == "organs":
            unexpected = values - set(t["source_index"].values())
            if unexpected:
                raise RuntimeError(f"unexpected label values {sorted(unexpected)}")
        out = t["lut"][data]
        img = nib.Nifti1Image(out, ct.affine)
        img.set_qform(ct.affine, code=1)
        img.set_sform(ct.affine, code=1)
        img.header.set_data_dtype(np.uint8)
        nib.save(img, tmp / "seg.nii.gz")
        atomic_copy(tmp / "seg.nii.gz", dest)
        present = [t["classes"][k - 1] for k in np.unique(out) if k > 0]
    return present


TRACER = dict(zip(inventory.study_id, inventory.tracer))


def top_gap_slices(mask, affine):
    z = np.flatnonzero(mask.any(axis=(0, 1)))
    if not len(z):
        return None
    superior_is_high = affine[2, 2] > 0
    return int(mask.shape[2] - 1 - z[-1]) if superior_is_high else int(z[0])


def run(studies, tasks=("organs", "salivary")):
    todo = [(s, task) for s in studies for task in tasks
            if not final_path(s, task).exists() and (RETRY_ERRORS or not error_path(s, task).exists())]
    n_done_before = sum(final_path(s, task).exists() for s in studies for task in tasks)
    print(f"{len(studies)} studies x {len(tasks)} tasks: {n_done_before} already done, {len(todo)} to run")
    t_run, n_ok, n_err = time.time(), 0, 0
    for i, (study_id, task) in enumerate(todo, start=1):
        t0 = time.time()
        row = {"timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"), "study_id": study_id,
               "tracer": TRACER.get(study_id, ""), "task": task, "gpu": GPU, "ts_version": TS_VERSION}
        try:
            present = segment(study_id, task)
            row.update(status="done", labels_present=";".join(present), n_labels=len(present))
            error_path(study_id, task).unlink(missing_ok=True)
            n_ok += 1
        except Exception as exc:
            row.update(status="error", error=f"{type(exc).__name__}: {exc}"[:300])
            error_path(study_id, task).write_text(traceback.format_exc())
            n_err += 1
        finally:
            row["seconds"] = round(time.time() - t0, 1)
            log(row)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        elapsed = time.time() - t_run
        eta_h = elapsed / i * (len(todo) - i) / 3600
        print(f"[{i}/{len(todo)}] {study_id} {task}: {row['status']} in {row['seconds']} s"
              f"{' (' + row['error'] + ')' if row['status'] == 'error' else ''} | ETA {eta_h:.1f} h", flush=True)
    print(f"finished: {n_ok} done, {n_err} errors, {(time.time() - t_run) / 60:.1f} min")

## 4. Test on 3 studies

The first three studies of the order (68Ga). They are written to the normal output folders, so the full run skips them. Check the timing per task, the classes found, and the overlay at the end of the section before launching the full run.

In [ ]:
TEST_STUDIES = ORDER[:3]
run(TEST_STUDIES)
pd.read_csv(LOG_CSV).query("study_id in @TEST_STUDIES")[["study_id", "task", "status", "seconds", "n_labels", "error"]]

## 4b. Organ model: fast (3 mm) vs standard (1.5 mm) on the test studies

The test studies are segmented again with the other organ model (`fast = not FAST_ORGANS`) into `compare_fast/`. Then, per study and class, the cell computes:
- the Dice coefficient between the two masks on the PET grid, $2|A \cap B| / (|A| + |B|)$;
- the volume difference $(V_\mathrm{fast} - V_\mathrm{standard}) / V_\mathrm{standard}$;
- the time per study of each model: from `run_log.csv` for the main model, measured here for the other.

The vertebrae are summarised as the union of all vertebrae and as the median Dice over the single vertebrae, because the per-vertebra labels can swap between neighbours without changing the marrow volume.

How to read it:
- Neither model is ground truth: the comparison measures **agreement**.
- At PET resolution one boundary voxel is 3-4 mm, so small or thin structures have a low Dice even when the two models agree on position: duodenum, aorta, single vertebrae, and a small bladder.
- Volume differences matter more than Dice for PBPK, since compartment volumes enter the model directly.
- Suggested rule, for a decision on 3 studies only: the fast model is acceptable if kidneys, liver and spleen have Dice ≥ 0.90 and |ΔV| ≤ 5% in every test study and the time gain is substantial. Otherwise keep the standard model.
- To switch, set `FAST_ORGANS`, delete the `organs/` masks of the test studies and the `labelmap_organs.json`, and rerun from the setup cell.

In [ ]:
OTHER_FAST = not FAST_ORGANS
CMP_DIR = OUT_DIR / "compare_fast" / ("organs_fast" if OTHER_FAST else "organs_standard")
CMP_DIR.mkdir(parents=True, exist_ok=True)
TIMING_CSV = OUT_DIR / "compare_fast" / "timing.csv"
timing = pd.read_csv(TIMING_CSV) if TIMING_CSV.exists() else pd.DataFrame(columns=["study_id", "fast", "seconds"])
for s in TEST_STUDIES:
    dest = CMP_DIR / f"{s}.nii.gz"
    if dest.exists() or not final_path(s, "organs").exists():
        continue
    t0 = time.time()
    try:
        segment(s, "organs", fast=OTHER_FAST, dest=dest)
        timing.loc[len(timing)] = [s, OTHER_FAST, round(time.time() - t0, 1)]
        timing.to_csv(TIMING_CSV, index=False)
        print(f"{s}: fast={OTHER_FAST} done in {time.time() - t0:.0f} s")
    except Exception as exc:
        print(f"{s}: fast={OTHER_FAST} failed: {type(exc).__name__}: {exc}")
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
def dice(a, b):
    s = a.sum() + b.sum()
    return 2 * np.logical_and(a, b).sum() / s if s else np.nan


rows = []
for s in TEST_STUDIES:
    main_p, other_p = final_path(s, "organs"), CMP_DIR / f"{s}.nii.gz"
    if not (main_p.exists() and other_p.exists()):
        continue
    img = nib.load(main_p)
    main, other = np.asarray(img.dataobj), np.asarray(nib.load(other_p).dataobj)
    std, fst = (other, main) if FAST_ORGANS else (main, other)
    voxel_ml = float(np.prod(img.header.get_zooms()[:3])) / 1000.0
    vert = [ORGAN_CLASSES.index(c) + 1 for c in VERTEBRAE]
    items = [(c, [k]) for k, c in enumerate(ORGAN_CLASSES, start=1)] + [("vertebrae (union)", vert)]
    for c, ks in items:
        a, b = np.isin(std, ks), np.isin(fst, ks)
        va, vb = a.sum() * voxel_ml, b.sum() * voxel_ml
        rows.append({"study_id": s, "class": c, "V_standard_ml": va, "V_fast_ml": vb,
                     "dV_%": 100 * (vb - va) / va if va > 0 else np.nan, "dice": dice(a, b)})
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT_DIR / "compare_fast" / "fast_vs_standard.csv", index=False)
print(f"{cmp.study_id.nunique()} studies compared")

is_vert = cmp["class"].isin(VERTEBRAE)
table = cmp[~is_vert].groupby("class", sort=False).agg(
    V_standard_ml_median=("V_standard_ml", "median"), dice_median=("dice", "median"), dice_min=("dice", "min"),
    dV_pct_median=("dV_%", "median"), dV_pct_max_abs=("dV_%", lambda x: x.abs().max()))
per_vert = cmp[is_vert & (cmp.V_standard_ml > 0)]
table.loc["vertebrae (median over single vertebrae)"] = [per_vert.V_standard_ml.median(), per_vert.dice.median(),
                                                         per_vert.dice.min(), per_vert["dV_%"].median(),
                                                         per_vert["dV_%"].abs().max()]
display(table.round(3))
display(cmp[~is_vert].pivot_table(index="class", columns="study_id", values="dice", sort=False).round(3))

In [ ]:
log_main = pd.read_csv(LOG_CSV).query("task == 'organs' and status == 'done' and study_id in @TEST_STUDIES")
t_main = log_main.groupby("study_id").seconds.last()
t_other = pd.read_csv(TIMING_CSV).groupby("study_id").seconds.last() if TIMING_CSV.exists() else pd.Series(dtype=float)
times = pd.DataFrame({"standard_s": t_other if FAST_ORGANS else t_main, "fast_s": t_main if FAST_ORGANS else t_other})
times["speed-up"] = times.standard_s / times.fast_s
display(times.round(1))

KEY = ["kidney_left", "kidney_right", "liver", "spleen"]
key = cmp[cmp["class"].isin(KEY)]
ok = bool(len(key)) and (key.dice >= 0.90).all() and (key["dV_%"].abs() <= 5).all()
print(f"kidneys, liver, spleen: min Dice {key.dice.min():.3f}, max |dV| {key['dV_%'].abs().max():.1f}% -> "
      f"{'fast model acceptable by the rule above' if ok else 'keep the standard model (rule not met)'}; "
      f"median speed-up x{times['speed-up'].median():.1f}")

## 4c. Salivary gland overlays of the test studies

Three views per test study, cropped to the top 220 mm of the CT and to ±110 mm around the centroid of the glands:
- a coronal slice through the centroid of the glands;
- a sagittal slice through each parotid (or through the submandibular gland of that side if the parotid is absent).

Parotids are shown in blue and submandibular glands in orange. The dashed line marks the top slice of the CT field of view, and the title gives the gap between each gland and that slice. A gland is flagged as truncated by the field of view if its most superior voxel is within `TRUNC_MARGIN_SLICES` = 2 slices of the top (section 6). The parotid reaches up to the external auditory canal, so in a field of view that starts at the skull base it is the gland most often cut.

The figures are saved in `qc/salivary_<study_id>.png`.

In [ ]:
GLAND_COLORS = {"parotid": "#2a78d6", "submandibular": "#eb6834"}
CROP_MM = 220
HALF_WIDTH_MM = 110


def salivary_views(study_id):
    ct = nib.load(ct_files[study_id])
    hu = np.asarray(ct.dataobj, dtype=np.float32)
    sal = np.asarray(nib.load(final_path(study_id, "salivary")).dataobj)
    dx, dy, dz = ct.header.get_zooms()[:3]
    nz = hu.shape[2]
    sup_high = ct.affine[2, 2] > 0
    n_crop = min(nz, int(round(CROP_MM / dz)))
    zsl = slice(nz - n_crop, nz) if sup_high else slice(0, n_crop)
    top_row = n_crop - 1 if sup_high else 0
    gaps = {c: top_gap_slices(sal == k, ct.affine) for k, c in enumerate(SALIVARY_CLASSES, start=1)}
    anygl = sal > 0
    x_c = int(np.round(np.nonzero(anygl)[0].mean())) if anygl.any() else hu.shape[0] // 2
    y_c = int(np.round(np.nonzero(anygl)[1].mean())) if anygl.any() else hu.shape[1] // 2
    hx, hy = int(round(HALF_WIDTH_MM / dx)), int(round(HALF_WIDTH_MM / dy))
    xsl = slice(max(x_c - hx, 0), min(x_c + hx + 1, hu.shape[0]))
    ysl = slice(max(y_c - hy, 0), min(y_c + hy + 1, hu.shape[1]))

    def x_of(side):
        for c in [f"parotid_gland_{side}", f"submandibular_gland_{side}"]:
            m = sal == SALIVARY_CLASSES.index(c) + 1
            if m.any():
                return int(np.round(np.nonzero(m)[0].mean())), c.split("_")[0]
        return (hu.shape[0] * (3 if side == "left" else 1)) // 4, "no gland"

    def overlay(lab2d):
        rgba = np.zeros(lab2d.shape + (4,))
        for k, c in enumerate(SALIVARY_CLASSES, start=1):
            rgba[lab2d == k] = (*plt.matplotlib.colors.to_rgb(GLAND_COLORS[c.split("_")[0]]), 0.5)
        return rgba

    views = [("coronal", f"y={y_c}", hu[xsl, y_c, zsl], sal[xsl, y_c, zsl], dz / dx)]
    for side in ["right", "left"]:
        x, what = x_of(side)
        views.append((f"sagittal {side}", f"x={x} ({what})", hu[x, ysl, zsl], sal[x, ysl, zsl], dz / dy))
    out = [(name, where, np.clip((img + 160) / 400, 0, 1).T, overlay(lab).transpose(1, 0, 2), asp)
           for name, where, img, lab, asp in views]
    return out, gaps, top_row


def show_salivary(study_id, save=True):
    views, gaps, top_row = salivary_views(study_id)
    fig, axes = plt.subplots(1, 3, figsize=(12, 5.2))
    for ax, (name, where, img, ov, asp) in zip(axes, views):
        ax.imshow(img, cmap="gray", origin="lower", aspect=asp)
        ax.imshow(ov, origin="lower", aspect=asp)
        ax.axhline(top_row, color="#52514e", lw=1.2, ls="--")
        ax.set_title(f"{name}, {where}", fontsize=9)
        ax.axis("off")
    gap_txt = ", ".join(f"{c.replace('_gland', '')}: {'absent' if g is None else f'{g} sl'}" for c, g in gaps.items())
    handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.6, label=n) for n, c in GLAND_COLORS.items()]
    handles.append(plt.Line2D([], [], color="#52514e", ls="--", label="top slice of the CT"))
    fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle(f"{study_id} ({TRACER.get(study_id, '')}) - gap to top slice: {gap_txt}", x=0.01, ha="left", fontsize=10)
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    if save:
        fig.savefig(OUT_DIR / "qc" / f"salivary_{study_id}.png", dpi=130)
    plt.show()


for s in TEST_STUDIES:
    if final_path(s, "salivary").exists():
        show_salivary(s)
    else:
        print(f"{s}: no salivary output (see run_log.csv / errors/)")

## 5. Full run

The cell can be re-run after a disconnection: it re-mounts nothing and repeats nothing, it only processes what is missing. A Colab session is limited in time, so the 597 studies × 2 tasks will usually take several sessions. The ETA printed after each study is based on the current session.

In [ ]:
run(ORDER)

In [ ]:
logdf = pd.read_csv(LOG_CSV)
last = logdf.sort_values("timestamp_utc").groupby(["study_id", "task"]).tail(1)
done = {task: sum(final_path(s, task).exists() for s in ORDER) for task in TASKS}
print("final files:", done, "of", len(ORDER), "studies")
display(last.groupby(["task", "status"]).seconds.describe()[["count", "50%", "max"]].rename(columns={"50%": "median s"}))
last[last.status == "error"].groupby(["task", "error"]).size().sort_values(ascending=False).head(20)

## 6. QC: volumes and flags

For every study with outputs:
- volume (mL) of each class;
- the axial range (first and last slice index) of each class;
- the number of connected components is not computed. Fragmented masks show up as volume outliers or in the overlays.

Flags. The thresholds are chosen for screening, not as physiological limits.

| flag | rule |
|---|---|
| `missing` | volume < 1 mL for kidneys, liver, spleen, heart, aorta |
| `cut` | kidneys, liver, spleen, heart, aorta, bladder: the mask touches the first or last axial slice of the CT (structure possibly truncated by the field of view) |
| `truncated (FOV)` | salivary gland present (≥ 0.5 mL) whose most superior voxel is within `TRUNC_MARGIN_SLICES` = 2 slices of the top slice of the CT (gap 0, 1 or 2 slices, i.e. ≤ 4-10 mm depending on the slice thickness): the gland is probably cut by the field of view and its volume underestimated |
| `outlier` | $\lvert \log V - \mathrm{median}\rvert > 3 \times 1.4826\,\mathrm{MAD}$ of $\log V$ over all studies, for organs and glands present (computed only with at least 20 studies) |
| `salivary absent (FOV)` | gland < 0.5 mL and neither `vertebrae_C1` nor `vertebrae_C2` segmented: the field of view probably stops below the glands |
| `salivary absent (seg)` | gland < 0.5 mL although C1 or C2 is present: segmentation failure, check visually |
| `salivary failed` | no `salivary` output (task error, e.g. no skull in the field of view) |

A missing or cut **kidney** matters most: kidneys are the dose-limiting organ, and a study with a cut kidney cannot be used for kidney PBPK fitting.

In [ ]:
QC_CLASSES = {"organs": ORGAN_CLASSES, "salivary": SALIVARY_CLASSES}
CORE = ["kidney_left", "kidney_right", "liver", "spleen", "heart", "aorta"]


def measure(study_id):
    row = {"study_id": study_id}
    for task, classes in QC_CLASSES.items():
        p = final_path(study_id, task)
        row[f"has_{task}"] = p.exists()
        if not p.exists():
            continue
        img = nib.load(p)
        data = np.asarray(img.dataobj)
        nz = data.shape[2]
        voxel_ml = float(np.prod(img.header.get_zooms()[:3])) / 1000.0
        counts = np.bincount(data.ravel(), minlength=len(classes) + 1)
        row["nz"] = nz
        dz = float(img.header.get_zooms()[2])
        for k, c in enumerate(classes, start=1):
            row[f"{c}_ml"] = counts[k] * voxel_ml
            if counts[k] and c in CORE + ["urinary_bladder"]:
                z = np.flatnonzero((data == k).any(axis=(0, 1)))
                row[f"{c}_cut"] = bool(z[0] == 0 or z[-1] == nz - 1)
            if counts[k] and c in SALIVARY_CLASSES:
                gap = top_gap_slices(data == k, img.affine)
                row[f"{c}_top_gap_slices"] = gap
                row[f"{c}_top_gap_mm"] = gap * dz
                row[f"{c}_truncated"] = gap <= TRUNC_MARGIN_SLICES
    return row


studies_with_output = [s for s in ORDER if any(final_path(s, t).exists() for t in TASKS)]
with ThreadPoolExecutor(8) as ex:
    qc = pd.DataFrame(list(ex.map(measure, studies_with_output)))
qc = inventory[["study_id", "subject", "tracer", "dx_mm", "dz_mm", "z_extent_mm", "ct_contrast"]].merge(qc, on="study_id")
for c in [f"{c}_cut" for c in CORE + ["urinary_bladder"]] + [f"{c}_truncated" for c in SALIVARY_CLASSES]:
    if c not in qc:
        qc[c] = False
    qc[c] = qc[c].astype("boolean").fillna(False)
print(f"{len(qc)} studies measured")

In [ ]:
def robust_outlier(v, min_studies=20):
    lv = np.log(v.where(v > 0))
    med = lv.median()
    mad = 1.4826 * (lv - med).abs().median()
    if lv.notna().sum() < min_studies or not mad > 0:
        return pd.Series(False, index=v.index)
    return (lv - med).abs() > 3 * mad


flags = []
for _, r in qc.iterrows():
    f = []
    if r.get("has_organs"):
        f += [f"{c} missing" for c in CORE if r.get(f"{c}_ml", 0) < 1.0]
        f += [f"{c} cut" for c in CORE + ["urinary_bladder"] if r.get(f"{c}_cut")]
    else:
        f.append("organs failed")
    if r.get("has_salivary"):
        c12 = r.get("vertebrae_C1_ml", 0) > 0 or r.get("vertebrae_C2_ml", 0) > 0
        for c in SALIVARY_CLASSES:
            if r.get(f"{c}_ml", 0) < 0.5:
                f.append(f"{c} absent ({'seg' if c12 else 'FOV'})")
            elif r.get(f"{c}_truncated"):
                f.append(f"{c} truncated (FOV)")
    else:
        f.append("salivary failed")
    flags.append(f)
qc["qc_flags"] = ["; ".join(f) for f in flags]

OUTLIER_CLASSES = ["kidney_left", "kidney_right", "liver", "spleen", "urinary_bladder", "heart", "aorta",
                   "stomach", "colon", "small_bowel"] + SALIVARY_CLASSES
for c in OUTLIER_CLASSES:
    if f"{c}_ml" in qc:
        out = robust_outlier(qc[f"{c}_ml"]).fillna(False)
        qc.loc[out, "qc_flags"] = qc.loc[out, "qc_flags"].where(qc.loc[out, "qc_flags"] == "", qc.loc[out, "qc_flags"] + "; ") + f"{c} outlier"
qc["n_flags"] = qc.qc_flags.map(lambda s: 0 if s == "" else s.count(";") + 1)

vol_cols = [f"{c}_ml" for c in ["kidney_left", "kidney_right", "liver", "spleen", "urinary_bladder", "heart", "aorta"]
            + SALIVARY_CLASSES if f"{c}_ml" in qc]
summary = qc.groupby("tracer")[vol_cols].median().T.round(1)
summary["present in % of studies"] = [100 * (qc[c] > (0.5 if "gland" in c else 1.0)).mean() for c in summary.index]
display(summary.round(1))

flag_list = pd.Series([x for f in flags for x in f] + [x for s in qc.qc_flags for x in s.split("; ") if x.endswith("outlier")])
flag_counts = flag_list.value_counts()
display(flag_counts.rename("studies").to_frame())

qc.to_csv(OUT_DIR / "qc" / "qc_volumes.csv", index=False)
qc.loc[qc.n_flags > 0, ["study_id", "tracer", "z_extent_mm", "n_flags", "qc_flags"]].to_csv(OUT_DIR / "qc" / "qc_flags.csv", index=False)
qc.loc[qc.n_flags > 0, ["study_id", "tracer", "z_extent_mm", "qc_flags"]].head(30)

### 6b. Salivary glands: truncation by the field of view and comparison with the literature

**Truncation.**
- Each gland of each study with a `salivary` output is classed as *present* (≥ 0.5 mL) or *absent*. Present glands are further split into *truncated* (gap to the top slice ≤ 2 slices) or *complete*.
- The fraction of truncated glands is reported per gland and per tracer, over the present glands.

**Literature reference values**, per gland:

| source | patients | method | parotid | submandibular |
|---|---|---|---|---|
| Okamoto et al., J Nucl Med 2017;58:445-450 (doi:10.2967/jnumed.116.178483) | 18, mCRPC, [177Lu]Lu-PSMA I&T | mass from PET/CT delineation, per gland | 19.1 ± 5.7 g [8.0-35.6] | 8.2 ± 1.9 g [4.2-14.3] |
| Scarpa et al., Eur J Nucl Med Mol Imaging 2017;44:788-800 (doi:10.1007/s00259-016-3609-9) | 10, mCRPC, [177Lu]Lu-PSMA-617 | volume from PET/CT delineation | 24.98 mL (max 34) | 8.63 mL (max 10.35) |

- Both rows are taken from Table 6 of the EANM dosimetry committee recommendations: Sjögreen Gleisner K et al., Eur J Nucl Med Mol Imaging 2022, doi:10.1007/s00259-022-05727-7. The original papers were not re-read.
- The same document (Table 5) gives the ICRP 89 reference mass of all salivary glands of the adult male, 85 g. This is the sum of parotid, submandibular and sublingual glands, so it is an upper bound for the sum of the four segmented glands.
- Okamoto gives masses. They are compared with our volumes assuming 1.0 g/mL; soft-tissue density 1.03-1.05 g/mL would change them by < 5%, well inside the spread.

Only complete glands enter the comparison. Truncated glands are shown separately: they should be smaller.

At PET resolution a submandibular gland of 8 mL has ~120 voxels of 4 × 4 × 4 mm, more than half of them on the boundary. A systematic difference of 10-20% from the literature is therefore within what the voxel size alone can produce. The literature values come from manual delineation on PET/CT, which has a similar limitation.

In [ ]:
long = []
for _, r in qc[qc.has_salivary.astype(bool)].iterrows():
    for c in SALIVARY_CLASSES:
        v = r.get(f"{c}_ml", 0.0)
        v = 0.0 if pd.isna(v) else v
        present = v >= 0.5
        long.append({"study_id": r.study_id, "tracer": r.tracer, "gland": c, "type": c.split("_")[0],
                     "side": c.split("_")[-1], "volume_ml": v, "present": present,
                     "truncated": bool(present and r.get(f"{c}_truncated", False)),
                     "top_gap_slices": r.get(f"{c}_top_gap_slices", np.nan), "top_gap_mm": r.get(f"{c}_top_gap_mm", np.nan)})
glands = pd.DataFrame(long)
glands.to_csv(OUT_DIR / "qc" / "qc_salivary_glands.csv", index=False)


def trunc_summary(g):
    p = g[g.present]
    return pd.Series({"studies": g.study_id.nunique(), "present": len(p), "absent": int((~g.present).sum()),
                      "truncated": int(p.truncated.sum()),
                      "truncated % of present": 100 * p.truncated.mean() if len(p) else np.nan})


display(glands.groupby("gland", sort=False).apply(trunc_summary, include_groups=False).round(1))
display(glands.groupby(["type", "tracer"]).apply(trunc_summary, include_groups=False).round(1))
both = glands[glands.present].groupby("study_id").truncated.any()
print(f"studies with at least one truncated gland: {int(both.sum())}/{glands.study_id.nunique()} "
      f"({100 * both.sum() / max(glands.study_id.nunique(), 1):.1f}%); overall truncated glands: "
      f"{int(glands.truncated.sum())}/{int(glands.present.sum())} present glands "
      f"({100 * glands.truncated.sum() / max(glands.present.sum(), 1):.1f}%)")

In [ ]:
LIT = pd.DataFrame([
    {"source": "Okamoto 2017 (via EANM 2022, Table 6)", "type": "parotid", "mean": 19.1, "sd": 5.7, "min": 8.0, "max": 35.6},
    {"source": "Okamoto 2017 (via EANM 2022, Table 6)", "type": "submandibular", "mean": 8.2, "sd": 1.9, "min": 4.2, "max": 14.3},
    {"source": "Scarpa 2017 (via EANM 2022, Table 6)", "type": "parotid", "mean": 24.98, "sd": np.nan, "min": np.nan, "max": 34.0},
    {"source": "Scarpa 2017 (via EANM 2022, Table 6)", "type": "submandibular", "mean": 8.63, "sd": np.nan, "min": np.nan, "max": 10.35},
])

rows = []
for typ in ["parotid", "submandibular"]:
    ok = LIT[(LIT.type == typ) & LIT.source.str.startswith("Okamoto")].iloc[0]
    for label, sel in [("complete", glands.present & ~glands.truncated), ("truncated", glands.truncated)]:
        v = glands.loc[sel & (glands.type == typ), "volume_ml"]
        rows.append({"type": typ, "glands": label, "n": len(v), "mean": v.mean(), "sd": v.std(), "median": v.median(),
                     "IQR": f"{v.quantile(0.25):.1f}-{v.quantile(0.75):.1f}" if len(v) else "-",
                     "min-max": f"{v.min():.1f}-{v.max():.1f}" if len(v) else "-",
                     "mean / Okamoto mean": v.mean() / ok["mean"],
                     "% inside Okamoto range": 100 * v.between(ok["min"], ok["max"]).mean() if len(v) else np.nan})
comparison = pd.DataFrame(rows).set_index(["type", "glands"])
comparison.to_csv(OUT_DIR / "qc" / "salivary_vs_literature.csv")
display(LIT)
comparison.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
rng_j = np.random.default_rng(0)
ROWS_Y = {"complete (this work)": 3, "truncated (this work)": 2, "Okamoto 2017": 1, "Scarpa 2017": 0}
for ax, typ in zip(axes, ["parotid", "submandibular"]):
    for label, sel, color in [("complete (this work)", glands.present & ~glands.truncated, "#2a78d6"),
                              ("truncated (this work)", glands.truncated, "#eb6834")]:
        v = glands.loc[sel & (glands.type == typ), "volume_ml"]
        y = ROWS_Y[label] + rng_j.uniform(-0.18, 0.18, len(v))
        ax.scatter(v, y, s=10, color=color, alpha=0.6, linewidths=0)
        if len(v):
            ax.plot([v.quantile(0.25), v.quantile(0.75)], [ROWS_Y[label]] * 2, color="#0b0b0b", lw=3)
            ax.plot(v.median(), ROWS_Y[label], "|", color="#0b0b0b", ms=14, mew=2)
    ok = LIT[(LIT.type == typ) & LIT.source.str.startswith("Okamoto")].iloc[0]
    ax.plot([ok["min"], ok["max"]], [1, 1], color="#52514e", lw=1)
    ax.plot([ok["mean"] - ok["sd"], ok["mean"] + ok["sd"]], [1, 1], color="#52514e", lw=4)
    ax.plot(ok["mean"], 1, "o", color="#52514e", ms=7)
    sc = LIT[(LIT.type == typ) & LIT.source.str.startswith("Scarpa")].iloc[0]
    ax.plot(sc["mean"], 0, "o", color="#52514e", ms=7)
    ax.plot(sc["max"], 0, "|", color="#52514e", ms=12, mew=2)
    ax.set_title(f"{typ} gland (per gland)", loc="left", fontsize=10)
    ax.set_xlabel("volume (mL); literature masses at 1.0 g/mL")
    ax.set_xlim(left=0)
axes[0].set_yticks(list(ROWS_Y.values()), list(ROWS_Y.keys()))
axes[0].grid(axis="y", visible=False)
axes[1].grid(axis="y", visible=False)
fig.text(0.01, 0.01, "This work: dots = glands, bar = IQR, tick = median. Okamoto: mean (dot), ±SD (thick), range (thin). "
         "Scarpa: mean (dot), max (tick).", fontsize=8, color="#52514e")
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.savefig(OUT_DIR / "qc" / "salivary_volumes_vs_literature.png", dpi=150)
plt.show()

**Kidney volume distribution.** Left vs right kidney volume per study. Each dot is a study, coloured by tracer; flagged studies are open circles. A kidney far from the diagonal or near zero points to a nephrectomy, an atrophic kidney, a cut mask or a segmentation error, and the overlays decide which.

In [ ]:
COLORS = {"18F": "#2a78d6", "68Ga": "#eb6834"}
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e4e3df", "grid.linewidth": 0.6})
fig, ax = plt.subplots(figsize=(5.5, 5))
kid_flag = qc.qc_flags.str.contains("kidney")
for tracer, c in COLORS.items():
    d = qc[(qc.tracer == tracer) & qc.has_organs.astype(bool)]
    k = kid_flag[d.index]
    ax.scatter(d.kidney_left_ml[~k], d.kidney_right_ml[~k], s=14, color=c, label=f"{tracer} (n = {len(d)})")
    ax.scatter(d.kidney_left_ml[k], d.kidney_right_ml[k], s=30, facecolor="#fcfcfb", edgecolor=c, linewidth=1.4)
lim = max(qc.kidney_left_ml.max(), qc.kidney_right_ml.max()) * 1.05
ax.plot([0, lim], [0, lim], color="#52514e", lw=0.8)
ax.set(xlim=(0, lim), ylim=(0, lim), xlabel="left kidney (mL)", ylabel="right kidney (mL)")
ax.legend(frameon=False, title="open circles: kidney flag", title_fontsize=8.5)
fig.tight_layout()
fig.savefig(OUT_DIR / "qc" / "kidney_volumes.png", dpi=150)
plt.show()

## 7. QC: coronal overlays

Random sample of studies (fixed seed), plus all studies with a kidney flag up to a maximum. Each study gets two coronal slices through the CT (soft-tissue window, W 400 / L 40 HU):
- through the centroid of the kidneys;
- through the centroid of the parotid glands, or 10 mm anterior to the kidney slice if the glands are absent.

Classes are grouped into 8 colours (categorical palette, fixed order): kidneys, liver, spleen, GI tract (stomach, duodenum, small bowel, colon), bladder, blood pool (heart, aorta), vertebrae, salivary glands. At PET resolution the edges are blocky: judge position and completeness, not the contour detail.

In [ ]:
GROUPS = {"kidneys": (["kidney_left", "kidney_right"], "#2a78d6"), "liver": (["liver"], "#eb6834"),
          "spleen": (["spleen"], "#1baf7a"), "GI tract": (["stomach", "duodenum", "small_bowel", "colon"], "#eda100"),
          "bladder": (["urinary_bladder"], "#e87ba4"), "blood pool": (["heart", "aorta"], "#008300"),
          "vertebrae": (VERTEBRAE, "#6250d6"), "salivary": (SALIVARY_CLASSES, "#e34948")}
N_RANDOM, N_FLAGGED_MAX, SEED = 8, 8, 20261101


def rgba_overlay(organs, salivary):
    rgba = np.zeros(organs.shape + (4,))
    for name, (classes, color) in GROUPS.items():
        src, cls = (salivary, SALIVARY_CLASSES) if name == "salivary" else (organs, ORGAN_CLASSES)
        if src is None:
            continue
        m = np.isin(src, [cls.index(c) + 1 for c in classes])
        rgba[m] = (*plt.matplotlib.colors.to_rgb(color), 0.45)
    return rgba


def coronal_panels(study_id):
    ct = nib.load(ct_files[study_id])
    hu = np.asarray(ct.dataobj, dtype=np.float32)
    organs = np.asarray(nib.load(final_path(study_id, "organs")).dataobj)
    sp = final_path(study_id, "salivary")
    salivary = np.asarray(nib.load(sp).dataobj) if sp.exists() else None
    dx, dy, dz = ct.header.get_zooms()[:3]
    kid = np.isin(organs, [1, 2])
    y_kid = int(np.round(np.nonzero(kid)[1].mean())) if kid.any() else hu.shape[1] // 2
    gl = np.isin(salivary, [1, 2]) if salivary is not None else None
    y_gl = int(np.round(np.nonzero(gl)[1].mean())) if gl is not None and gl.any() else None
    if y_gl is None:
        y_gl = int(np.clip(y_kid + round(10 / dy), 0, hu.shape[1] - 1))
    out = []
    for y in [y_kid, y_gl]:
        ct_sl = np.clip((hu[:, y, :] + 160) / 400, 0, 1).T
        ov = rgba_overlay(organs[:, y, :], salivary[:, y, :] if salivary is not None else None).transpose(1, 0, 2)
        out.append((ct_sl, ov, dz / dx, y))
    return out


rng = np.random.default_rng(SEED)
done_organs = [s for s in ORDER if final_path(s, "organs").exists()]
sample = list(rng.choice(done_organs, size=min(N_RANDOM, len(done_organs)), replace=False))
flagged = [s for s in qc.loc[qc.qc_flags.str.contains("kidney"), "study_id"] if s not in sample][:N_FLAGGED_MAX]


def overlay_figure(studies, title, filename):
    if not studies:
        print(f"{title}: no studies")
        return
    ncol = 4
    nrow = int(np.ceil(2 * len(studies) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.6 * ncol, 6.2 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for i, s in enumerate(studies):
        for j, (ct_sl, ov, aspect, y) in enumerate(coronal_panels(s)):
            ax = axes.ravel()[2 * i + j]
            ax.imshow(ct_sl, cmap="gray", origin="lower", aspect=aspect)
            ax.imshow(ov, origin="lower", aspect=aspect)
            r = qc.set_index("study_id").loc[s] if s in set(qc.study_id) else None
            note = "" if r is None or r.qc_flags == "" else "\n" + r.qc_flags[:60]
            ax.set_title(f"{s[5:21]} {TRACER.get(s, '')} y={y}{' (kidneys)' if j == 0 else ''}{note}", fontsize=7.5)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.6, label=n) for n, (_, c) in GROUPS.items()]
    fig.legend(handles=handles, loc="lower center", ncol=8, frameon=False, fontsize=9)
    fig.suptitle(title, x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0.03, 1, 0.98))
    fig.savefig(OUT_DIR / "qc" / filename, dpi=110)
    plt.show()


overlay_figure(sample, f"Random sample of {len(sample)} studies (seed {SEED})", "overlays_random.png")
overlay_figure(flagged, f"Studies with a kidney flag ({len(flagged)} shown)", "overlays_kidney_flags.png")

## 8. Outputs

All outputs are on Drive under `psma_segmentations/`. For the local workspace, download these files to `data/processed/psma/segmentations_qc/`; the masks stay on Drive:
- `qc/qc_volumes.csv`, `qc/qc_flags.csv`, `qc/qc_salivary_glands.csv` and `qc/salivary_vs_literature.csv`;
- the two label maps;
- `run_log.csv`.

Before the masks are used for PBPK or dosimetry:
- review the flagged studies;
- record the excluded studies and the reason, e.g. in a column of the inventory;
- check the small structures (salivary glands, duodenum) in the overlays.